In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

Project Root: d:\Live Master\Projects\travel-dynamic-pricing-rl


# Deep Q-Network (DQN)

## Objective

Tabular Q-Learning performs well only when the state space is small. As the number of states increases, storing and updating a Q-table becomes impractical.

A Deep Q-Network (DQN) replaces the Q-table with a neural network that approximates the Q-values for each action.

Advantages of DQN:

- Handles larger state spaces.
- Generalizes to unseen states.
- Learns non-linear state-action relationships.
- Forms the foundation of modern reinforcement learning algorithms.

In [2]:
import torch

from src.dqn_agent import DQN

model = DQN()

state = torch.tensor([[50.0, 30.0]])

model(state)

tensor([[ 0.7673, -0.6732, -0.6066,  1.1969, -1.1874]],
       grad_fn=<AddmmBackward0>)

## Experience Replay

Training directly from consecutive experiences causes highly correlated updates, making neural network training unstable.

Experience Replay addresses this issue by storing transitions in a replay buffer.

Each transition consists of:

- Current State
- Action
- Reward
- Next State
- Done Flag

During training, random mini-batches are sampled from this buffer, improving data efficiency and stabilizing learning.

In [3]:
from src.replay_buffer import ReplayBuffer

buffer = ReplayBuffer()

buffer.add([50, 30], 2, 100, [49, 29], False)

len(buffer)

1

In [4]:
buffer.sample(1)

[([50, 30], 2, 100, [49, 29], False)]

## Replay Buffer Operations

The replay buffer supports three primary operations:

1. Store transitions after every interaction.
2. Randomly sample mini-batches.
3. Automatically discard the oldest experiences when the buffer reaches its maximum capacity.

These operations enable more stable and efficient deep reinforcement learning.

## DQN Agent

The Deep Q-Network itself is responsible only for estimating Q-values.

The DQN Agent manages the learning process by:

- Selecting actions using an epsilon-greedy policy.
- Storing the neural network.
- Managing the optimizer.
- Computing the training loss.
- Updating the network parameters.

In [5]:
from src.dqn_agent import DQNAgent

agent = DQNAgent()

print("Initial epsilon:", agent.epsilon)

state = [50, 30]

for _ in range(5):
    print(agent.choose_action(state))

agent.decay_epsilon()

print("Updated epsilon:", agent.epsilon)

Initial epsilon: 1.0
0
2
2
1
2
Updated epsilon: 0.995


## Epsilon-Greedy Exploration in DQN

The DQN Agent balances exploration and exploitation using an epsilon-greedy policy.

- Initially, epsilon is set to 1.0, encouraging exploration.
- During training, epsilon gradually decreases.
- As epsilon approaches its minimum value, the agent increasingly exploits the learned Q-values produced by the neural network.

## DQN Learning Process

Unlike tabular Q-Learning, DQN updates the neural network using mini-batches sampled from the replay buffer.

For each sampled transition:

1. Predict the current Q-value.
2. Compute the target Q-value.
3. Calculate the Mean Squared Error (MSE) loss.
4. Perform backpropagation.
5. Update the neural network parameters.

This process enables the network to approximate the optimal action-value function.

In [6]:
from src.dqn_agent import DQNAgent
from src.replay_buffer import ReplayBuffer

agent = DQNAgent()
buffer = ReplayBuffer()

print("Initial epsilon:", agent.epsilon)
print("Replay Buffer:", len(buffer))

Initial epsilon: 1.0
Replay Buffer: 0


## Mini-Batch Gradient Descent

The replay buffer provides randomly sampled transitions that are used to compute the loss and update the neural network.

Unlike tabular Q-Learning, the DQN agent learns through gradient descent, allowing it to generalize across unseen states.

## Target Network

One challenge in Deep Q-Networks is that the network is trying to predict values that are simultaneously changing during training.

To stabilize learning, DQN introduces a **Target Network**.

Instead of computing target Q-values using the same network being updated, a second network with frozen parameters is used.

The target network is periodically synchronized with the online network, reducing oscillations and improving convergence.

In [7]:
from src.dqn_agent import DQNAgent

agent = DQNAgent()

print(agent.model)
print()
print(agent.target_model)

DQN(
  (network): Sequential(
    (0): Linear(in_features=2, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=5, bias=True)
  )
)

DQN(
  (network): Sequential(
    (0): Linear(in_features=2, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=5, bias=True)
  )
)


## Why a Target Network?

Using the online network to compute both current and target Q-values can lead to unstable updates because the target is constantly changing.

The target network provides a stable reference for computing Bellman targets and is synchronized periodically with the online network.

## Training the DQN Agent

The DQN agent interacts with the environment over multiple simulated booking seasons.

For each interaction:

1. The current transition is stored in the replay buffer.
2. Mini-batches are sampled once sufficient experiences have been collected.
3. The neural network is updated using gradient descent.
4. The target network is periodically synchronized.
5. The exploration rate gradually decreases over time.

The trained model parameters are saved for later evaluation.

In [8]:
import numpy as np

rewards = np.load("../models/dqn_rewards.npy")

print("Episodes:", len(rewards))

print("Average Reward:", rewards.mean())

Episodes: 500
Average Reward: 1518.2
